### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [1]:
%pip install numpy scikit-learn

### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [3]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [4]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [5]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [6]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [7]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [8]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [9]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [10]:
# tfidfvect.vocabulary_['cocoliso']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [11]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [12]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [13]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [14]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [15]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [16]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ])

Después vemos a qué documentos corresponden

In [17]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  4703, 10870,  4333])

Obtenemos los 5 documentos más similares:

In [18]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [19]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [20]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [21]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

MultinomialNB()

Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [22]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [23]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


---
## Desarrollo del Desafío 1

### 1. Similaridad entre documentos

Elegimos 5 documentos de train al azar y buscamos sus 5 vecinos más parecidos (similaridad coseno). El primer resultado es el mismo documento, por eso lo saltamos.

In [24]:
rng = np.random.default_rng(42)
docs_azar = rng.choice(len(newsgroups_train.data), 5, replace=False)
nombres = newsgroups_train.target_names

for i in docs_azar:
    cs = cosine_similarity(X_train[i], X_train)[0]
    top = np.argsort(cs)[::-1][1:6]
    print(f"\n=== Doc {i} | clase: {nombres[y_train[i]]} ===")
    print(newsgroups_train.data[i][:250].replace('\n', ' '))
    for j in top:
        print(f"  {j:5d} sim={cs[j]:.3f} [{nombres[y_train[j]]}] "
              f"{newsgroups_train.data[j][:100]!r}")
    print(f"  -> {np.mean(y_train[top] == y_train[i]):.0%} de los 5 vecinos comparten clase")


=== Doc 8754 | clase: talk.religion.misc ===
 /(hudson) /If someone inflicts pain on themselves, whether they enjoy it or not, they /are hurting themselves.  They may be permanently damaging their body.  That is true.  It is also none of your business.    Some people may also reason that by rea
   6552 sim=0.490 [talk.religion.misc] '\nIf I have a habit that I really want to break, and I am willing to\nmake whatever sacrifice I need t'
  10613 sim=0.481 [talk.religion.misc] '/(hudson)\n/Yes you do.  Who is to say that it is immoral for onesself to experience\n/pain or to be h'
   3616 sim=0.465 [talk.religion.misc] '\nAnd I maintain:\n\nSome people do not want to enter into the light and the knowledge that\nthey alone '
   8726 sim=0.460 [talk.politics.mideast] '\n\n[After a small refresh Hasan got on the track again.]\n\n\n\n\n   |>    |> I get the impression Hasan r'
   3902 sim=0.459 [talk.religion.misc] '\n\n\nAre you your own master?  Do you have any habits that you cannot break?

**Qué vemos:**
- Funciona bien cuando el tema es claro. Los documentos de religión y de armas tienen 80% de vecinos de su clase.
- Muchos vecinos son del mismo autor o del mismo hilo (mismo estilo, mismas citas).
- Falla con textos cortos y generales. El documento de Windows tiene 0%: sus vecinos solo comparten palabras comunes como "hello" y "application".
- Las clases parecidas se mezclan, por ejemplo `comp.sys.mac.hardware` con `comp.sys.ibm.pc.hardware`.

La similaridad mide palabras en común, no la etiqueta.

### 2. Clasificación por prototipos (1-NN)

A cada documento de test le damos la clase del documento de train más parecido. TF-IDF ya está normalizado, así que el producto punto es la similaridad coseno. Lo hacemos de 1000 en 1000 documentos para usar menos memoria.

In [25]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target

y_pred_1nn = []
for s in range(0, X_test.shape[0], 1000):
    S = (X_test[s:s+1000] @ X_train.T).toarray()
    y_pred_1nn.append(y_train[S.argmax(axis=1)])
y_pred_1nn = np.concatenate(y_pred_1nn)

print("F1 macro 1-NN :", f1_score(y_test, y_pred_1nn, average='macro'))
print("Accuracy 1-NN :", (y_pred_1nn == y_test).mean())

F1 macro 1-NN : 0.5049911553681621
Accuracy 1-NN : 0.5088953797132236


**Resultado:** F1 macro = 0.505.

Es un buen punto de partida porque no hay entrenamiento. Pero un solo vecino decide la clase, y ese vecino a veces es un documento general o de una clase parecida.

### 3. Naïve Bayes

Empezamos con el modelo de la clase (TF-IDF + MultinomialNB). Después buscamos mejores parámetros con una grilla:
- Vectorizador: `stop_words`, `sublinear_tf`, `max_df` (no cambiamos `ngram_range`).
- Modelo: MultinomialNB o ComplementNB, y `alpha`.

Elegimos los parámetros con validación cruzada (CV) en train. Así no usamos test para decidir. Test se usa solo al final. Tarda unos minutos.

In [26]:
import tempfile
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Modelo base de la clase, como referencia
f1_base = f1_score(y_test, MultinomialNB().fit(X_train, y_train).predict(X_test), average='macro')
print("F1 macro base (TF-IDF por defecto + MultinomialNB, alpha=1):", f1_base)

pipe = Pipeline([('vec', TfidfVectorizer()), ('clf', MultinomialNB())],
                memory=tempfile.mkdtemp())

grid = {
    'vec__stop_words': [None, 'english'],
    'vec__sublinear_tf': [False, True],
    'vec__max_df': [1.0, 0.5],
    'clf': [MultinomialNB(), ComplementNB()],
    'clf__alpha': [0.003, 0.01, 0.03, 0.1, 0.3, 1.0],
}
gs = GridSearchCV(pipe, grid, scoring='f1_macro', n_jobs=-1,
                  cv=StratifiedKFold(4, shuffle=True, random_state=0))
gs.fit(newsgroups_train.data, y_train)

res = pd.DataFrame(gs.cv_results_)
res['modelo'] = res['param_clf'].map(lambda c: type(c).__name__)
cols = ['modelo', 'param_clf__alpha', 'param_vec__stop_words',
        'param_vec__sublinear_tf', 'param_vec__max_df', 'mean_test_score']
print("\nTop 10 configuraciones (F1 macro en CV sobre train):")
print(res.sort_values('mean_test_score', ascending=False)[cols].head(10).to_string())

print("\nMejor configuración:", gs.best_params_)
print("F1 macro CV (train):", gs.best_score_)
y_pred_best = gs.best_estimator_.predict(newsgroups_test.data)
print("F1 macro TEST      :", f1_score(y_test, y_pred_best, average='macro'))

F1 macro base (TF-IDF por defecto + MultinomialNB, alpha=1): 0.5854345727938506

Top 10 configuraciones (F1 macro en CV sobre train):
          modelo  param_clf__alpha param_vec__stop_words  param_vec__sublinear_tf  param_vec__max_df  mean_test_score
81  ComplementNB               0.3                  None                     True                1.0         0.760198
85  ComplementNB               0.3                  None                     True                0.5         0.759969
83  ComplementNB               0.3               english                     True                1.0         0.759266
87  ComplementNB               0.3               english                     True                0.5         0.759266
84  ComplementNB               0.3                  None                    False                0.5         0.758538
82  ComplementNB               0.3               english                    False                1.0         0.758155
86  ComplementNB               0.3      

In [27]:
from sklearn.base import clone

# Mejor configuración de cada modelo, evaluada en test
for nombre in ['MultinomialNB', 'ComplementNB']:
    mejor = res[res['modelo'] == nombre].nlargest(1, 'mean_test_score').iloc[0]
    modelo = clone(pipe).set_params(**mejor['params']).fit(newsgroups_train.data, y_train)
    f1 = f1_score(y_test, modelo.predict(newsgroups_test.data), average='macro')
    print(f"{nombre:14s} alpha={mejor['param_clf__alpha']:<6} CV={mejor['mean_test_score']:.4f}  TEST={f1:.4f}")

MultinomialNB  alpha=0.01   CV=0.7544  TEST=0.6844
ComplementNB   alpha=0.3    CV=0.7602  TEST=0.6990


**Resultado:**
- Modelo base: 0.585.
- MultinomialNB con `alpha=0.01`: 0.684. Bajar `alpha` ayuda mucho: con TF-IDF los pesos son pequeños y `alpha=1` suaviza demasiado.
- ComplementNB con `alpha=0.3`: **0.699** (el mejor). Usa los documentos de las otras clases para estimar cada clase, y suele ser más estable.
- `stop_words`, `sublinear_tf` y `max_df` casi no cambian el resultado (en el top 10 la diferencia es menor a 0.003). TF-IDF ya reduce el peso de las palabras comunes.
- El F1 en CV (0.760) es más alto que en test (0.699). Probablemente porque test está separado por fecha, y CV mezcla mensajes del mismo hilo. Por eso CV es optimista.

### 4. Similaridad entre palabras

Transponemos la matriz: ahora cada palabra es un vector con sus pesos TF-IDF en todos los documentos. Usamos `stop_words='english'` y `min_df=5` para quitar palabras raras, que dominan el ranking sin tener sentido. Elegimos 5 palabras a mano.

In [28]:
tfidf_pal = TfidfVectorizer(stop_words='english', min_df=5)
X_pal = tfidf_pal.fit_transform(newsgroups_train.data)
XT = X_pal.T.tocsr()                    # matriz término x documento
vocab = tfidf_pal.vocabulary_
inv_vocab = {i: w for w, i in vocab.items()}
print("Matriz término-documento:", XT.shape)

for palabra in ['car', 'god', 'space', 'israel', 'window']:
    cs = cosine_similarity(XT[vocab[palabra]], XT)[0]
    top = np.argsort(cs)[::-1][1:6]
    print(f"{palabra:8s}", [(inv_vocab[i], round(float(cs[i]), 2)) for i in top])

Matriz término-documento: (17797, 11314)
car      [('cars', 0.19), ('dealer', 0.18), ('civic', 0.16), ('loan', 0.16), ('owner', 0.15)]
god      [('jesus', 0.28), ('bible', 0.27), ('christ', 0.27), ('faith', 0.25), ('existence', 0.25)]
space    [('nasa', 0.32), ('shuttle', 0.28), ('exploration', 0.23), ('aeronautics', 0.22), ('cfa', 0.22)]
israel   [('israeli', 0.42), ('arab', 0.38), ('syrian', 0.37), ('lebanon', 0.36), ('lebanese', 0.33)]
window   [('manager', 0.3), ('decoration', 0.29), ('xcreatewindow', 0.28), ('xmapwindow', 0.27), ('override', 0.23)]


**Resultado:** dos palabras son parecidas si aparecen en los mismos documentos. Los vecinos tienen sentido:
- *car* → cars, dealer, civic, loan
- *god* → jesus, bible, christ, faith
- *space* → nasa, shuttle, exploration
- *israel* → israeli, arab, syrian, lebanon
- *window* → manager, xcreatewindow, xmapwindow. Es vocabulario de X Window System: en este corpus domina ese sentido y no el de "ventana" normal. Esto muestra la polisemia: el corpus decide el significado.

Las similaridades son bajas (0.15 a 0.42) porque los vectores son muy grandes y casi vacíos.